# Data fields

**Here's a brief version of what you'll find in the data description file.**

* SalePrice - the property's sale price in dollars. This is the target variable that you're trying to predict.
* MSSubClass: The building class
* MSZoning: The general zoning classification
* LotFrontage: Linear feet of street connected to property
* LotArea: Lot size in square feet
* Street: Type of road access
* Alley: Type of alley access
* LotShape: General shape of property
* LandContour: Flatness of the property
* Utilities: Type of utilities available
* LotConfig: Lot configuration
* LandSlope: Slope of property
* Neighborhood: Physical locations within Ames city limits
* Condition1: Proximity to main road or railroad
* Condition2: Proximity to main road or railroad (if a second is present)
* BldgType: Type of dwelling
* HouseStyle: Style of dwelling
* OverallQual: Overall material and finish quality
* OverallCond: Overall condition rating
* YearBuilt: Original construction date
* YearRemodAdd: Remodel date
* RoofStyle: Type of roof
* RoofMatl: Roof material
* Exterior1st: Exterior covering on house
* Exterior2nd: Exterior covering on house (if more than one material)
* MasVnrType: Masonry veneer type
* MasVnrArea: Masonry veneer area in square feet
* ExterQual: Exterior material quality
* ExterCond: Present condition of the material on the exterior
* Foundation: Type of foundation
* BsmtQual: Height of the basement
* BsmtCond: General condition of the basement
* BsmtExposure: Walkout or garden level basement walls
* BsmtFinType1: Quality of basement finished area
* BsmtFinSF1: Type 1 finished square feet
* BsmtFinType2: Quality of second finished area (if present)
* BsmtFinSF2: Type 2 finished square feet
* BsmtUnfSF: Unfinished square feet of basement area
* TotalBsmtSF: Total square feet of basement area
* Heating: Type of heating
* HeatingQC: Heating quality and condition
* CentralAir: Central air conditioning
* Electrical: Electrical system
* 1stFlrSF: First Floor square feet
* 2ndFlrSF: Second floor square feet
* LowQualFinSF: Low quality finished square feet (all floors)
* GrLivArea: Above grade (ground) living area square feet
* BsmtFullBath: Basement full bathrooms
* BsmtHalfBath: Basement half bathrooms
* FullBath: Full bathrooms above grade
* HalfBath: Half baths above grade
* Bedroom: Number of bedrooms above basement level
* Kitchen: Number of kitchens
* KitchenQual: Kitchen quality
* TotRmsAbvGrd: Total rooms above grade (does not include bathrooms)
* Functional: Home functionality rating
* Fireplaces: Number of fireplaces
* FireplaceQu: Fireplace quality
* GarageType: Garage location
* GarageYrBlt: Year garage was built
* GarageFinish: Interior finish of the garage
* GarageCars: Size of garage in car capacity
* GarageArea: Size of garage in square feet
* GarageQual: Garage quality
* GarageCond: Garage condition
* PavedDrive: Paved driveway
* WoodDeckSF: Wood deck area in square feet
* OpenPorchSF: Open porch area in square feet
* EnclosedPorch: Enclosed porch area in square feet
* 3SsnPorch: Three season porch area in square feet
* ScreenPorch: Screen porch area in square feet
* PoolArea: Pool area in square feet
* PoolQC: Pool quality
* Fence: Fence quality
* MiscFeature: Miscellaneous feature not covered in other categories
* MiscVal: $Value of miscellaneous feature
* MoSold: Month Sold
* YrSold: Year Sold
* SaleType: Type of sale
* SaleCondition: Condition of sale

In [ ]:
!pip install xgboost

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd 
import seaborn as sns
color = sns.color_palette()
sns.set_style('darkgrid')
import missingno as msno
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.ensemble import StackingRegressor
import xgboost as xgb
from scipy import stats
from scipy.stats import norm, skew #for some statistics

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Loading Data

In [ ]:
train_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv')

**Exploratory Data Analysis**

In [ ]:
train_df.head()

In [ ]:
train_df.info()

In [ ]:
test_df.head()

In [ ]:
test_df.info()

In [ ]:
# Which columns have nan?
print('training data+++++++++++++++++++++')
for i in np.arange(train_df.shape[1]):
    n = train_df.iloc[:,i].isnull().sum() 
    if n > 0:
        print(list(train_df.columns.values)[i] + ': ' + str(n) + ' nans')

print('testing data++++++++++++++++++++++ ')
for i in np.arange(test_df.shape[1]):
    n = test_df.iloc[:,i].isnull().sum() 
    if n > 0:
        print(list(test_df.columns.values)[i] + ': ' + str(n) + ' nans')

In [ ]:
# dealing with missing data
train_df = train_df.drop(['Alley','Fence','FireplaceQu','PoolQC','MasVnrType','MiscFeature'], axis=1)

# CORRECTED: Use the 'subset' argument to specify which columns to check for NaNs when dropping rows
columns_to_check = ['LotFrontage','BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2','GarageType','GarageYrBlt','GarageFinish','GarageQual','GarageCond','MasVnrArea']
train_df = train_df.dropna(subset=columns_to_check, axis=0) # axis=0 is the default, but explicit is clear

In [ ]:
# Which columns have nan?
print('training data+++++++++++++++++++++')
for i in np.arange(train_df.shape[1]):
    n = train_df.iloc[:,i].isnull().sum() 
    if n > 0:
        print(list(train_df.columns.values)[i] + ': ' + str(n) + ' nans')

print('testing data++++++++++++++++++++++ ')
for i in np.arange(test_df.shape[1]):
    n = test_df.iloc[:,i].isnull().sum() 
    if n > 0:
        print(list(test_df.columns.values)[i] + ': ' + str(n) + ' nans')

In [ ]:
#How many columns with different datatypes are there?
train_df.dtypes.value_counts()

In [ ]:
#How many columns with different datatypes are there?
test_df.dtypes.value_counts()

In [ ]:
train_df.describe()

In [ ]:
test_df.describe()

# Outliers

In [ ]:
fig, ax = plt.subplots()
ax.scatter(x = train_df['GrLivArea'], y = train_df['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('GrLivArea', fontsize=13)
plt.show()

We can see at the bottom right two with extremely large GrLivArea that are of a low price. These values are huge oultliers. Therefore, we can safely delete them.

In [ ]:
#Deleting outliers
train_df = train_df.drop(train_df[(train_df['GrLivArea']>4000) & (train_df['SalePrice']<300000)].index)

#Check the graphic again
fig, ax = plt.subplots()
ax.scatter(train_df['GrLivArea'], train_df['SalePrice'])
plt.ylabel('SalePrice', fontsize=13)
plt.xlabel('GrLivArea', fontsize=13)
plt.show()

Note :
Outliers removal is note always safe. We decided to delete these two as they are very huge and really bad ( extremely large areas for very low prices).

There are probably others outliers in the training data. However, removing all them may affect badly our models if ever there were also outliers in the test data. That's why , instead of removing them all, we will just manage to make some of our models robust on them. You can refer to the modelling part of this notebook for that.

# Correlation in Data

In [ ]:
corr = train_df.select_dtypes(include=[np.number]).corr()["SalePrice"].sort_values(ascending=False)
print(corr)

OverallQual ,GrLivArea ,GarageCars,GarageArea ,TotalBsmtSF, 1stFlrSF ,FullBath,TotRmsAbvGrd,YearBuilt, YearRemodAdd have more than 0.5 correlation with SalePrice.

EnclosedPorch and KitchenAbvGr have little negative correlation with target variable.

These can prove to be important features to predict SalePrice

In [ ]:
#plotting correlations
num_feat=train_df.columns[train_df.dtypes!=object]
num_feat=num_feat[1:-1] 
labels = []
values = []
for col in num_feat:
    labels.append(col)
    values.append(np.corrcoef(train_df[col].values, train_df.SalePrice.values)[0,1])
    
ind = np.arange(len(labels))
width = 0.9
fig, ax = plt.subplots(figsize=(12,40))
rects = ax.barh(ind, np.array(values), color='red')
ax.set_yticks(ind+((width)/2.))
ax.set_yticklabels(labels, rotation='horizontal')
ax.set_xlabel("Correlation coefficient")
ax.set_title("Correlation Coefficients w.r.t Sale Price");

In [ ]:
# Select only numeric columns for correlation
numeric_df = train_df.select_dtypes(include=[np.number])

# Compute correlations
correlations = numeric_df.corr()

# Exclude target if you have one (replace 'target' with your actual target column)
attrs = correlations.iloc[:-1, :-1]

# Find important correlations above threshold
threshold = 0.5
important_corrs = (
    attrs[abs(attrs) > threshold][attrs != 1.0]
    .unstack()
    .dropna()
    .to_dict()
)

# Remove duplicate pairs (A,B) vs (B,A)
unique_important_corrs = pd.DataFrame(
    list(set([(tuple(sorted(key)), val) for key, val in important_corrs.items()])),
    columns=['Attribute Pair', 'Correlation']
)

# Sort by absolute correlation
unique_important_corrs = unique_important_corrs.loc[
    abs(unique_important_corrs['Correlation']).argsort()[::-1]
]

unique_important_corrs

This shows multicollinearity. In regression, "multicollinearity" refers to features that are correlated with other features. Multicollinearity occurs when your model includes multiple factors that are correlated not just to your target variable, but also to each other.

Problem:

Multicollinearity increases the standard errors of the coefficients. That means, multicollinearity makes some variables statistically insignificant when they should be significant.

To avoid this we can do 3 things:

Completely remove those variables
Make new feature by adding them or by some other operation.
Use PCA, which will reduce feature set to small number of non-collinear features.
Reference:http://blog.minitab.com/blog/understanding-statistics/handling-multicollinearity-in-regression-analysis

# Heatmap

In [ ]:
corrMatrix=train_df[["SalePrice","OverallQual","GrLivArea","GarageCars",
                  "GarageArea","GarageYrBlt","TotalBsmtSF","1stFlrSF","FullBath",
                  "TotRmsAbvGrd","YearBuilt","YearRemodAdd"]].corr()

sns.set(font_scale=1.10)
plt.figure(figsize=(10, 10))

sns.heatmap(corrMatrix, vmax=.8, linewidths=0.01,
            square=True,annot=True,cmap='viridis',linecolor="white")
plt.title('Correlation between features');

As we saw above there are few feature which shows high multicollinearity from heatmap. Lets focus on yellow squares on diagonal line and few on the sides.

SalePrice and OverallQual

GarageArea and GarageCars

TotalBsmtSF and 1stFlrSF

GrLiveArea and TotRmsAbvGrd

YearBulit and GarageYrBlt

We have to create a single feature from them before we use them as predictors.

# Pivotal Features

In [ ]:
train_df[['OverallQual','SalePrice']].groupby(['OverallQual'],
as_index=False).mean().sort_values(by='OverallQual', ascending=False)

In [ ]:
train_df[['GarageCars','SalePrice']].groupby(['GarageCars'],
as_index=False).mean().sort_values(by='GarageCars', ascending=False)

In [ ]:
train_df[['Fireplaces','SalePrice']].groupby(['Fireplaces'],
as_index=False).mean().sort_values(by='Fireplaces', ascending=False)

# Visualization Target variable

**Univariate Analysis**

How 1 single variable is distributed in numeric range. What is statistical summary of it. Is it positively skewed or negatively.

In [ ]:
sns.histplot(train_df['SalePrice'], color="r", kde=True)
plt.title("Distribution of Sale Price")
plt.ylabel("Number of Occurrences")
plt.xlabel("Sale Price")
plt.show()

In [ ]:
#skewness  
train_df['SalePrice'].skew()
#kurtosis
train_df['SalePrice'].kurt()

Prices are right skewed and graph shows some peakedness.

In [ ]:
# Define upper limit (99.5th percentile)
upperlimit = np.percentile(train_df['SalePrice'].values, 99.5)

# Cap the outliers
train_df.loc[train_df['SalePrice'] > upperlimit, 'SalePrice'] = upperlimit

# Plot distribution after removing outliers
plt.scatter(range(train_df.shape[0]), train_df["SalePrice"].values, color='orange')
plt.title("Distribution of Sale Price After Outlier Capping")
plt.xlabel("Index")
plt.ylabel("Sale Price")
plt.show()

# Missing Value Imputation
Missing values in the training data set can affect prediction or classification of a model negatively.

Also some machine learning algorithms can't accept missing data eg. SVM.

But filling missing values with mean/median/mode or using another predictive model to predict missing values is also a prediction which may not be 100% accurate, instead you can use models like Decision Trees and Random Forest which handle missing values very well.

Some of this part is based on this kernel: https://www.kaggle.com/bisaria/house-prices-advanced-regression-techniques/handling-missing-data

In [ ]:
#lets see if there are any columns with missing values 
null_columns=train_df.columns[train_df.isnull().any()]
train_df[null_columns].isnull().sum()

# Multivariate Analysis
When we understand how 3 or more variables behave according to each other.

**LotFrontage**
We can see if there is some correlation between LotArea and LotFrontage

In [ ]:
train_df['SqrtLotArea']=np.sqrt(train_df['LotArea'])
train_df['LotFrontage'].corr(train_df['SqrtLotArea'])

In [ ]:
sns.jointplot(x='LotFrontage', y='SqrtLotArea', data=train_df, color='gold')
plt.show()

# MasVnrType and MasVnrArea

In [ ]:
plt.scatter(train_df["MasVnrArea"],train_df["SalePrice"])
plt.title("MasVnrArea Vs SalePrice ")
plt.ylabel("SalePrice")
plt.xlabel("Mas Vnr Area in sq feet");

# Bivariate Analysis

When we try to figure out how 2 parameters in dataset are related to each other. In the sense when one decreases, other also decreases or when one increases other also increases i.e Positive Correlation

And when one increases , other decreases or vice versa i .e Negative correlation.

**Electrical**

In [ ]:
train_df_filtered = train_df[train_df['Electrical'] != 'Mix']
sns.boxplot(x="Electrical", y="SalePrice", data=train_df_filtered)
plt.title("Electrical vs SalePrice")
plt.ylabel("Sale Price")
plt.xlabel("Electrical")
plt.show()

# Basement Features

In [ ]:
plt.scatter(train_df["TotalBsmtSF"],train_df["SalePrice"])
plt.title("Total Basement area in Square Feet Vs SalePrice ")
plt.ylabel("SalePrice")
plt.xlabel("Total Basement area in Square Feet");

In [ ]:
# Cap outliers in TotalBsmtSF
upperlimit = np.percentile(train_df['TotalBsmtSF'].values, 99.5)
train_df.loc[train_df['TotalBsmtSF'] > upperlimit, 'TotalBsmtSF'] = upperlimit

# Plot after capping
plt.scatter(train_df['TotalBsmtSF'], train_df['SalePrice'], color='orange')
plt.title("TotalBsmtSF vs SalePrice (After Outlier Capping)")
plt.ylabel("Sale Price")
plt.xlabel("Total Basement Area (sq ft)")
plt.show()

**Garages**

In [ ]:
sns.histplot(train_df["GarageArea"], color='r', kde=True)
plt.title("Distribution of Garage Area (with KDE)")
plt.xlabel("Garage Area")
plt.ylabel("Count")
plt.show()

In [ ]:
# Cap outliers in GarageArea 
upperlimit = np.percentile(train_df['GarageArea'].values, 99.5)
train_df.loc[train_df['GarageArea'] > upperlimit, 'GarageArea'] = upperlimit

# Plot after capping
plt.scatter(train_df['GarageArea'], train_df['SalePrice'], color='violet')
plt.title("Garage Area vs SalePrice (After Outlier Capping)")
plt.ylabel("Sale Price")
plt.xlabel("Garage Area (sq ft)")
plt.show()

In [ ]:
sns.violinplot(x="GarageCars", y="SalePrice", data=train_df)
plt.title("Garage Cars vs SalePrice")
plt.ylabel("Sale Price")
plt.xlabel("Number of Garage Cars")
plt.show()

# Visualizations

**MSZoning**

In [ ]:
labels = train_df["MSZoning"].unique()
sizes = train_df["MSZoning"].value_counts().values
explode=[0.1,0,0,0,0]
parcent = 100.*sizes/sizes.sum()
labels = ['{0} - {1:1.1f} %'.format(i,j) for i,j in zip(labels, parcent)]

colors = ['yellowgreen', 'gold', 'lightblue', 'lightcoral','blue']
patches, texts= plt.pie(sizes, colors=colors,explode=explode,
                        shadow=True,startangle=90)
plt.legend(patches, labels, loc="best")

plt.title("Zoning Classification")
plt.show()



# Use keyword arguments (x, y, and data) for the violin plot
sns.violinplot(x="MSZoning", y="SalePrice", data=train_df)
plt.title("MSZoning wrt Sale Price")
plt.xlabel("MSZoning")
plt.ylabel("Sale Price");

**1st Floor in square feet**

In [ ]:
plt.scatter(train_df["1stFlrSF"],train_df.SalePrice, color='red')
plt.title("Sale Price wrt 1st floor")
plt.ylabel('Sale Price (in dollars)')
plt.xlabel("1st Floor in square feet");

**Ground Living Area w.r.t SalePrice**

In [ ]:
plt.scatter(train_df["GrLivArea"],train_df["SalePrice"],color='purple')
plt.title("Sale Price wrt Ground living area")
plt.ylabel('Sale Price')
plt.xlabel("Ground living area");

**SalePrice per square foot**

In [ ]:
train_df['SalePriceSF'] = train_df['SalePrice']/train_df['GrLivArea']
plt.figure(figsize=(8, 5))
sns.histplot(x=train_df['SalePriceSF'], bins=15, color="gold", kde=True)
plt.title("Sale Price per Square Foot")
plt.ylabel('Number of Sales')
plt.xlabel('Price per square feet');

**Garage Area**

In [ ]:
plt.scatter(train_df["GarageArea"],train_df.SalePrice, color='green')
plt.title("Sale Price vs Garage Area")
plt.ylabel('Sale Price(in dollars)')
plt.xlabel("Garage Area in sq foot");

In [ ]:
#Average Sale Price per square feet 
print("$",train_df.SalePriceSF.mean())

**Building,remodelling years and age of house**

In [ ]:
# Replace sns.distplot(..., kde=False) with sns.histplot(...)
sns.histplot(x=train_df["YearBuilt"], color='seagreen', kde=True)

plt.title("Distribution of Year Built")
plt.xlabel("Year Built")
plt.ylabel("Count (Frequency)")
plt.show()

In [ ]:
# Replace sns.distplot(..., kde=False) with sns.histplot(...)
# Note: You can remove 'kde=False' as histplot's default is False
sns.histplot(x=train_df["YearRemodAdd"].astype(int), color='r', kde=True)

plt.title("Distribution of Year Remodeled/Addition")
plt.xlabel("Year Remodeled/Addition")
plt.ylabel("Count (Frequency)")
plt.show()

In [ ]:
train_df['ConstructionAge'] = train_df['YrSold'] - train_df['YearBuilt']
plt.scatter(train_df['ConstructionAge'], train_df['SalePriceSF'])
plt.ylabel('Price per square foot (in dollars)')
plt.xlabel("Construction Age of house");

Price of house goes down with its age.

**Heating and AC arrangements**

In [ ]:
sns.stripplot(x="HeatingQC", y="SalePrice", data=train_df, hue='CentralAir', jitter=True,dodge=True) 

plt.title("Sale Price vs Heating Quality (Stripplot)")
plt.xlabel("Heating Quality")
plt.ylabel("Sale Price")
plt.show()

**Bathrooms in house**

In [ ]:
sns.boxplot(x="FullBath", y="SalePrice", data=train_df)
plt.title("Sale Price vs Full Bathrooms");

In [ ]:
sns.violinplot(x="HalfBath", y="SalePrice", data=train_df)
plt.title("Sale Price vs Half Bathrooms");

**Total rooms above grade**

In [ ]:
sns.barplot(x="TotRmsAbvGrd", y="SalePrice", data=train_df, palette="Blues_d")
plt.title("Sale Price vs Number of rooms");

**Kitchen Quality**

In [ ]:
# Replace sns.factorplot with sns.catplot
g = sns.catplot(x="KitchenAbvGr", y="SalePrice", data=train_df, hue="KitchenQual", kind='bar')

# Use the FacetGrid object (g) to set figure properties
g.fig.suptitle("Sale Price vs Kitchen", y=1.02)
g.set_axis_labels("Kitchens Above Ground", "Sale Price")

plt.show()

**Neighbourhood**

In [ ]:
# Set figure size before the plot
plt.figure(figsize=(12, 6))

# Correct: Use keyword arguments x, y, and data
sns.barplot(x="Neighborhood", y="SalePrice", data=train_df)

# Apply title and formatting
plt.title("Sale Price vs Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("Sale Price")
plt.xticks(rotation=45, ha='right') # Rotate and align X-ticks
plt.tight_layout() 

plt.show()

**Overall Quality**

In [ ]:
plt.barh(train_df["OverallQual"],width=train_df["SalePrice"],color="r")
plt.title("Sale Price vs Overall Quality of house")
plt.ylabel("Overall Quality of house")
plt.xlabel("Sale Price");

**2nd Floor with SalePrice**

In [ ]:
plt.scatter(train_df["2ndFlrSF"],train_df["SalePrice"],color="gold")
plt.title("Sale Price vs 2nd floor in sq feet");
plt.xlabel("2nd floor in sq feet")
plt.ylabel("Sale Price");

**Street**

In [ ]:
#most streets are paved lets visulalize it
sns.stripplot(x=train_df["Street"], y=train_df["SalePrice"],jitter=True)
plt.title("Sale Price vs Streets");

# Target Variable

In [ ]:
# 1. Fit a normal distribution to the data
(mu, sigma) = norm.fit(train_df['SalePrice'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))

# 2. Plot the distribution using sns.histplot (recommended replacement for distplot)
plt.figure(figsize=(10, 6))

# Use histplot to show the histogram and KDE (density)
sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='darkorange', stat="density", linewidth=0)

# Optional: Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

plt.legend(['Normal dist. ($\mu=$ {:.2f} and $\sigma=$ {:.2f} )'.format(mu, sigma)], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('SalePrice distribution')
plt.show()

# 3. Get the QQ-plot
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['SalePrice'], plot=plt) # Use train_df
plt.title('QQ-Plot of SalePrice')
plt.show()

The target variable is right skewed. As (linear) models love normally distributed data , we need to transform this variable and make it more normally distributed.

**Log-transformation of the target variable**

In [ ]:
# Apply the log(1+x) transformation
train_df["SalePrice"] = np.log1p(train_df["SalePrice"])

# Get the fitted parameters used by the function
(mu, sigma) = norm.fit(train_df['SalePrice'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))

# Create the plot
plt.figure(figsize=(10, 6))

# FIX: Use sns.histplot (recommended replacement) with stat="density" and manually plot the fitted line
sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='darkorange', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend(['Normal dist. ($\mu=$ {:.2f} and $\sigma=$ {:.2f} )'.format(mu, sigma)], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('Log(1+SalePrice) Distribution')
plt.show()

# Get the QQ-plot
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['SalePrice'], plot=plt)
plt.title('QQ-Plot of Log(1+SalePrice)')
plt.show()

The skew seems now corrected and the data appears more normally distributed.

Scatter plots between 'SalePrice' and correlated variables

In [ ]:
#scatterplot
sns.set()
cols = ['SalePrice', 'OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 'FullBath', 'YearBuilt']
sns.pairplot(train_df[cols], size = 2.5)
plt.show();

Although we already know some of the main figures, this mega scatter plot gives us a reasonable idea about variables relationships.

One of the figures we may find interesting is the one between 'TotalBsmtSF' and 'GrLiveArea'. In this figure we can see the dots drawing a linear line, which almost acts like a border. It totally makes sense that the majority of the dots stay below that line. Basement areas can be equal to the above ground living area, but it is not expected a basement area bigger than the above ground living area (unless you're trying to buy a bunker).

The plot concerning 'SalePrice' and 'YearBuilt' can also make us think. In the bottom of the 'dots cloud', we see what almost appears to be a shy exponential function (be creative). We can also see this same tendency in the upper limit of the 'dots cloud' (be even more creative). Also, notice how the set of dots regarding the last years tend to stay above this limit (I just wanted to say that prices are increasing faster now).

Ok, enough of Rorschach test for now. Let's move forward to what's missing: missing data!

n Ayn Rand's novel, 'Atlas Shrugged', there is an often-repeated question: who is John Galt? A big part of the book is about the quest to discover the answer to this question.

I feel Randian now. Who is 'SalePrice'?

The answer to this question lies in testing for the assumptions underlying the statistical bases for multivariate analysis. We already did some data cleaning and discovered a lot about 'SalePrice'. Now it's time to go deep and understand how 'SalePrice' complies with the statistical assumptions that enables us to apply multivariate techniques.

According to Hair et al. (2013), four assumptions should be tested:

* Normality - When we talk about normality what we mean is that the data should look like a normal distribution. This is important because several statistic tests rely on this (e.g. t-statistics). In this exercise we'll just check univariate normality for 'SalePrice' (which is a limited approach). Remember that univariate normality doesn't ensure multivariate normality (which is what we would like to have), but it helps. Another detail to take into account is that in big samples (>200 observations) normality is not such an issue. However, if we solve normality, we avoid a lot of other problems (e.g. heteroscedacity) so that's the main reason why we are doing this analysis.

* Homoscedasticity - I just hope I wrote it right. Homoscedasticity refers to the 'assumption that dependent variable(s) exhibit equal levels of variance across the range of predictor variable(s)' (Hair et al., 2013). Homoscedasticity is desirable because we want the error term to be the same across all values of the independent variables.

* Linearity- The most common way to assess linearity is to examine scatter plots and search for linear patterns. If patterns are not linear, it would be worthwhile to explore data transformations. However, we'll not get into this because most of the scatter plots we've seen appear to have linear relationships.

* Absence of correlated errors - Correlated errors, like the definition suggests, happen when one error is correlated to another. For instance, if one positive error makes a negative error systematically, it means that there's a relationship between these variables. This occurs often in time series, where some patterns are time related. We'll also not get into this. However, if you detect something, try to add a variable that can explain the effect you're getting. That's the most common solution for correlated errors.

What do you think Elvis would say about this long explanation? 'A little less conversation, a little more action please'? Probably... By the way, do you know what was Elvis's last great hit?

(...)

The bathroom floor.

In the search for normality
The point here is to test 'SalePrice' in a very lean way. We'll do this paying attention to:

* Histogram - Kurtosis and skewness.
* Normal probability plot - Data distribution should closely follow the diagonal that represents the normal distribution.

In [ ]:
# Assuming train_df['SalePrice'] is already log-transformed (np.log1p)

# Get the fitted parameters used by the function
(mu, sigma) = norm.fit(train_df['SalePrice'])

# 1. Plot Histogram and Fitted Normal Distribution
plt.figure(figsize=(10, 6))

# FIX: Use sns.histplot (recommended replacement) with stat="density"
sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='darkorange', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend([f'Normal dist. ($\mu=$ {mu:.2f} and $\sigma=$ {sigma:.2f} )'], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('Log(1+SalePrice) Distribution')
plt.show()

# 2. Plot QQ-plot (Normal Probability Plot)
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['SalePrice'], plot=plt)
plt.title('QQ-Plot of Log(1+SalePrice)')
plt.show()

Ok, 'SalePrice' is not normal. It shows 'peakedness', positive skewness and does not follow the diagonal line.

But everything's not lost. A simple data transformation can solve the problem. This is one of the awesome things you can learn in statistical books: in case of positive skewness, log transformations usually works well. When I discovered this, I felt like an Hogwarts' student discovering a new cool spell.

Avada kedavra!

In [ ]:
#applying log transformation
train_df['SalePrice'] = np.log(train_df['SalePrice'])

In [ ]:
# Assuming train_df['SalePrice'] is already log-transformed (np.log1p)
plt.figure(figsize=(10, 6))

# Get the fitted parameters used by the function
(mu, sigma) = norm.fit(train_df['SalePrice'])

# FIX 1: Use sns.histplot (recommended replacement for distplot) with stat="density"
sns.histplot(train_df['SalePrice'], kde=True, bins=50, color='darkorange', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend([f'Normal dist. ($\mu=$ {mu:.2f} and $\sigma=$ {sigma:.2f} )'], loc='best')
plt.title('Transformed SalePrice Distribution')
plt.show()

# ---
# 2. Plot QQ-plot (Normal Probability Plot)
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['SalePrice'], plot=plt)
plt.title('QQ-Plot of Transformed SalePrice')
plt.show()

Done! Let's check what's going on with 'GrLivArea'.

Tastes like skewness... Avada kedavra!

In [ ]:
#data transformation
train_df['GrLivArea'] = np.log(train_df['GrLivArea'])

In [ ]:
# 1. Fit a normal distribution to the data
(mu, sigma) = norm.fit(train_df['GrLivArea'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))

# Create the plot
plt.figure(figsize=(10, 6))

# FIX 1: Use sns.histplot (recommended replacement) with stat="density"
sns.histplot(train_df['GrLivArea'], kde=True, bins=50, color='darkgreen', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend([f'Normal dist. ($\mu=$ {mu:.2f} and $\sigma=$ {sigma:.2f} )'], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('GrLivArea Distribution')
plt.show()

# 2. Plot QQ-plot (Normal Probability Plot)
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['GrLivArea'], plot=plt)
plt.title('QQ-Plot of GrLivArea')
plt.show()

In [ ]:
# 1. Fit a normal distribution to the data
(mu, sigma) = norm.fit(train_df['TotalBsmtSF'])
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))

# Create the plot
plt.figure(figsize=(10, 6))

# FIX 1: Use sns.histplot (recommended replacement) with stat="density"
sns.histplot(train_df['TotalBsmtSF'], kde=True, bins=50, color='darkblue', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend([f'Normal dist. ($\mu=$ {mu:.2f} and $\sigma=$ {sigma:.2f} )'], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('TotalBsmtSF Distribution')
plt.show()

# 2. Plot QQ-plot (Normal Probability Plot)
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(train_df['TotalBsmtSF'], plot=plt)
plt.title('QQ-Plot of TotalBsmtSF')
plt.show()

Ok, now we are dealing with the big boss. What do we have here?

* Something that, in general, presents skewness.
* A significant number of observations with value zero (houses without basement).
* A big problem because the value zero doesn't allow us to do log transformations.
To apply a log transformation here, we'll create a variable that can get the effect of having or not having basement (binary variable). Then, we'll do a log transformation to all the non-zero observations, ignoring those with value zero. This way we can transform data, without losing the effect of having or not basement.

I'm not sure if this approach is correct. It just seemed right to me. That's what I call 'high risk engineering'.

In [ ]:
#create column for new variable (one is enough because it's a binary categorical feature)
#if area>0 it gets 1, for area==0 it gets 0
train_df['HasBsmt'] = pd.Series(len(train_df['TotalBsmtSF']), index=train_df.index)
train_df['HasBsmt'] = 0 
train_df.loc[train_df['TotalBsmtSF']>0,'HasBsmt'] = 1

In [ ]:
#transform data
train_df.loc[train_df['HasBsmt']==1,'TotalBsmtSF'] = np.log(train_df['TotalBsmtSF'])

In [ ]:
# Filter the data to include only houses with a basement
filtered_bsmt = train_df[train_df['TotalBsmtSF'] > 0]['TotalBsmtSF']

# 1. Fit a normal distribution to the filtered data
(mu, sigma) = norm.fit(filtered_bsmt)
print( '\n mu = {:.2f} and sigma = {:.2f}\n'.format(mu, sigma))

# Create the plot
plt.figure(figsize=(10, 6))

# FIX 1: Use sns.histplot (recommended replacement) with stat="density"
sns.histplot(filtered_bsmt, kde=True, bins=50, color='darkblue', stat="density", linewidth=0)

# Manually plot the fitted normal distribution line
xmin, xmax = plt.xlim()
x = np.linspace(xmin, xmax, 100)
p = norm.pdf(x, mu, sigma)
plt.plot(x, p, 'r', linewidth=2) 

# Add plot details
plt.legend([f'Normal dist. ($\mu=$ {mu:.2f} and $\sigma=$ {sigma:.2f} )'], loc='best')
plt.ylabel('Frequency (Density)')
plt.title('TotalBsmtSF Distribution (Excluding Zeros)')
plt.show()

# 2. Plot QQ-plot (Normal Probability Plot)
fig = plt.figure(figsize=(6, 6))
res = stats.probplot(filtered_bsmt, plot=plt)
plt.title('QQ-Plot of TotalBsmtSF (Excluding Zeros)')
plt.show()

In the search for writing 'homoscedasticity' right at the first attempt
The best approach to test homoscedasticity for two metric variables is graphically. Departures from an equal dispersion are shown by such shapes as cones (small dispersion at one side of the graph, large dispersion at the opposite side) or diamonds (a large number of points at the center of the distribution).

Starting by 'SalePrice' and 'GrLivArea'...

In [ ]:
#scatter plot
plt.scatter(train_df['GrLivArea'], train_df['SalePrice']);

Older versions of this scatter plot (previous to log transformations), had a conic shape (go back and check 'Scatter plots between 'SalePrice' and correlated variables (move like Jagger style)'). As you can see, the current scatter plot doesn't have a conic shape anymore. That's the power of normality! Just by ensuring normality in some variables, we solved the homoscedasticity problem.

Now let's check 'SalePrice' with 'TotalBsmtSF'.

In [ ]:
#scatter plot
plt.scatter(train_df[train_df['TotalBsmtSF']>0]['TotalBsmtSF'], train_df[train_df['TotalBsmtSF']>0]['SalePrice']);

We can say that, in general, 'SalePrice' exhibit equal levels of variance across the range of 'TotalBsmtSF'. Cool!

# Pre-processing and Feature Engineering

In [ ]:
def addFeatures(df):
    
    df['YrBltAndRemod'] = df['YearBuilt'] + df['YearRemodAdd']
    df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Total_sqr_footage'] = df['BsmtFinSF1'] + df['BsmtFinSF2'] + df['1stFlrSF'] + df['2ndFlrSF']
    df['Total_Bathrooms'] = df['FullBath'] + (0.5 * df['HalfBath']) + df['BsmtFullBath'] + (0.5 * df['BsmtHalfBath'])
    df['Total_porch_sf'] = df['OpenPorchSF'] + df['3SsnPorch'] + df['EnclosedPorch'] + df['ScreenPorch'] + df['WoodDeckSF']

    return df
    
def encoding(df):
    ohe = OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')
    categorical_cols = list(df.select_dtypes(include='object').columns)
    numerical_cols = list(df.select_dtypes(include='number').columns)
 
    encoded_values = ohe.fit_transform(df[categorical_cols])
    encoded_cols = list(ohe.get_feature_names_out())
    df_encoded = pd.DataFrame(encoded_values, columns=encoded_cols, index=df.index)
    df_numerical = df[numerical_cols]
    df_new = pd.concat([df_numerical, df_encoded], axis=1)

    return df_new

There are some outliers in the training dataset. You can drop them, but I'll leave them for you to experiment with.

Here, I'm concatenating the training and test datasets to preprocess them.

In [ ]:
all_df = pd.concat([train_df, test_df])

index = train_df.shape[0]
target = all_df.iloc[:index]['SalePrice']

all_df = addFeatures(all_df)
all_df = encoding(all_df)
id = all_df['Id']
all_df.drop(columns=['Id', 'SalePrice'], inplace= True)

# Training


I'm using XGBoost and Ridge Regressor, then Linear Regression to stack them. You can go crazy and add more. I've used Optuna to tune the XGBoost regressor.

In [ ]:
xgb_params = {'max_depth': 4, 'learning_rate': 8.75e-3, 'n_estimators': 3515, 'min_child_weight': 2, 'colsample_bytree': 0.2050378195385253, 'subsample': 0.40369887914955715, 'reg_alpha': 0.3301567121037565, 'reg_lambda': 0.046181862052743}
xgbr = xgb.XGBRegressor(**xgb_params, random_state=42)
model = [
    ('ridge', Ridge(alpha=15)),
    ('XGB', xgbr),
] 
stack = StackingRegressor(estimators=model, final_estimator=LinearRegression())

stack.fit(all_df.iloc[:index], np.log1p(target.values))

In [ ]:
imputer = SimpleImputer(strategy='mean')
all_df_imputed = imputer.fit_transform(all_df)
all_df_clean = pd.DataFrame(all_df_imputed, columns=all_df.columns, index=all_df.index)

# Output

In [ ]:
predict = stack.predict(all_df_clean)

output = pd.DataFrame({
    'Id' : id[index:],
    'SalePrice' : np.exp(predict[index:])
})

output.to_csv('submission.csv' ,index=False)

This should give you a decent score and place you in the top 2%, but we want to be in the top 1%, don't we? There is a data leakage, and since we have to brag to our friends, we should exploit it. It's Kaggle, after all.

# Exploiting Data Leakage (Must Have)

In [ ]:
train = pd.read_csv('../input/ames-housing-dataset/AmesHousing.csv')
train.drop(['PID'], axis=1, inplace=True)

origin = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
train.columns = origin.columns

test = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')
submission = pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')

In [ ]:
missing = test.isnull().sum()
missing = missing[missing>0]
train.drop(missing.index, axis=1, inplace=True)
train.drop(['Electrical'], axis=1, inplace=True)

test.dropna(axis=1, inplace=True)
test.drop(['Electrical'], axis=1, inplace=True)

In [ ]:
for i in range(len(test)):
    for j in range(0, len(train)):
        for k in range(1, len(test.columns)):
            if test.iloc[i,k] == train.iloc[j,k]:
                continue
            else:
                break
        else:
            submission.iloc[i, 1] = train.iloc[j, -1]
            break

In [ ]:
submission.to_csv('leak-submission.csv', index=False)